In [1]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


In [2]:
## loading model from dotenv

import os
from dotenv import load_dotenv,dotenv_values
load_dotenv()

m2 = os.getenv("MODEL_3")
print(m2)

llama3.1:8b


In [3]:
#model = OllamaLLM(model="llama3.1:8b")    ##directly use model name without dotenv
model = OllamaLLM(model=m2)

In [4]:
template = """
You are an expert in recommendation for the restaurant named "Pizza Place" and thus will be asked about various type of restaurants and where the best type of particular food will be available, etc.
You might also be expected to plan some retaurants based on rough iternary and thu, you would be expected to recommend multiple restaurants at a time. 

Example:
User: " We are going to a bar and then would be going to a have dinner after it, 
can you recommend some bar that bar that ha good beer and BBQ restaurant?"
Expected Workflow: First search for the firt type of restaurant and then look for the other and recommend them."

You should not be overly formal but at the ame time, keep your answers simple and easy to understand as the users typically prefer short answer with a touch of familiarity,
as if they are having a conversation with friend.
Do not hallucinate and give at-most 3 recomendation with the information such a the place's name .

Here are some relevant reviews: {reviews}

Here is the quesstion to answer: {questions}
"""

prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

##test
chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":"What is the best pizza place in town?"})

"You're lookin' for some good pizza, huh? Based on reviews, I'd say Lafamilia is the way to go! They've got amazing reviews and are considered the best in town."

In [5]:
## Write in a python script
'''
while True:
    question = input("Ask your question (q to quit):")
    if question == "q":
        break
    
    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})
    
'''

'\nwhile True:\n    question = input("Ask your question (q to quit):")\n    if question == "q":\n        break\n\n    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})\n\n'

In [6]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
import os
import pandas as pd

In [7]:
dir = "data/csv/RRr.csv"
df = pd.read_csv(dir)

In [8]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:v1")

In [9]:
db_location = "./chrome_langchain_db"
add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    ids = []
    
    for i, row in df.iterrows():
        document = Document(
            page_content = row["Title"] + " " + row["Review"],
            metadata = {"rating": row["Rating"], "date":row["Date"]},
            id= str(i)
        )
        ids.append(str(i))
        documents.append(document)
    
    
vector_store = Chroma(
    collection_name = "restaurant_reviews",
    persist_directory=db_location,
    embedding_function = embeddings
)
        
if add_documents:
    vector_store.add_documents(documents=documents,ids = ids)


In [10]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k":5}
)

In [11]:
question = "What is the best pizza available here?"

reviews = retriever.invoke(question)

chain.invoke({"reviews":reviews,"questions":question})

"The best pizza in town, hands down! Based on the reviews, I'd recommend the following:\n\n1. **The Signature Pepperoni Pizza** - It's a fan favorite, and for good reason. The pepperoni curls up into little cups of deliciousness, and the sauce-to-cheese ratio is just right.\n2. **The Fig and Prosciutto Pizza** - This one's a game-changer. The sweetness of the figs pairs perfectly with the saltiness of the prosciutto, and the staff's beer pairing suggestions will take it to the next level.\n3. **The Nutella and Banana Pizza** - Yep, you read that right! This dessert pizza is surprisingly good, and perfect for those with a sweet tooth.\n\nGive any of these a try, and you won't be disappointed!"